In [5]:
#!/usr/bin/env python3
# =========================================================
# 🏛️ LEGAL SUMMARIZATION WITH HSLN (PLAIN PYTORCH)
# No HuggingFace PreTrainedModel - Pure PyTorch loading
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM
)
import evaluate
from collections import defaultdict
from tqdm import tqdm

# =========================
# DEVICE
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥 Using device: {device}")

# =========================
# PATHS (EDIT IF NEEDED)
# =========================
ROLE_MODEL_PATH = "best_RR_model"
LED_MODEL_PATH = "LED"
TRAIN_JSON = "train.json"
TEST_JSON = "test.json"
IMPORTANCE_FILE = "importance_scores.json"
OUTPUT_FILE = "final_results_mcs_hsln.json"

TOP_K = 150
MAX_SAMPLES = 3000
BATCH_SIZE = 16

# Rhetorical role labels
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "ANALYSIS", "STA", "PRE_RELIED", "PRE_NOT_RELIED", "RATIO", "RPC", "NONE"
]
NUM_LABELS = 13

# =========================
# NLTK
# =========================
try:
    nltk.data.find("tokenizers/punkt")
except:
    nltk.download("punkt")

# =========================================================
# HSLN MODEL (PURE PYTORCH - NO HUGGINGFACE)
# =========================================================
class PrototypeAttention(nn.Module):
    """Prototype-based multi-head attention"""
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h = heads
        self.dh = dim // heads
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.o = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, p):
        B, T, D = x.shape
        C = p.size(0)
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(C, self.h, self.dh)
        V = self.v(p).view(C, self.h, self.dh)
        Q = F.normalize(Q, dim=-1)
        K = F.normalize(K, dim=-1)
        
        outs = []
        for h in range(self.h):
            a = (Q[:, :, h] @ K[:, h].T).softmax(-1)
            a = self.dropout(a)
            outs.append(a @ V[:, h])
        out = torch.cat(outs, -1)
        return self.ln(x + self.dropout(self.o(out)))

class RPL(nn.Module):
    """Representation Prototype Learning"""
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))
        
    def forward(self, h):
        return F.normalize(h, dim=-1) @ F.normalize(self.proto, dim=-1).T

class RTM(nn.Module):
    """Rhetorical Transition Model"""
    def __init__(self, num_labels, rtm_lambda):
        super().__init__()
        self.A = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.rtm_lambda = rtm_lambda
        
    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(
                lp[:, t-1].unsqueeze(1) + self.A.log_softmax(-1), -1
            )
            logits[:, t] += self.rtm_lambda * tr
        return logits

class HSLNModel(nn.Module):
    """HSLN Model - Pure PyTorch (No HuggingFace)"""
    
    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13, 
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        
        self.embedding_dim = embedding_dim
        self.lstm_hidden = lstm_hidden
        self.num_labels = num_labels
        
        # Layers
        self.att = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm = nn.LSTM(
            embedding_dim, 
            lstm_hidden, 
            2, 
            bidirectional=True, 
            batch_first=True,
            dropout=dropout
        )
        self.cls = nn.Linear(lstm_hidden * 2, num_labels)
        self.rpl = RPL(lstm_hidden * 2, num_labels)
        self.alpha = nn.Parameter(torch.tensor(2.0))
        self.rtm = RTM(num_labels, rtm_lambda)
        
        # Prototypes buffer
        self.register_buffer('prototypes', torch.randn(num_labels, embedding_dim))
        
    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        
        if lengths is not None:
            pck = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        
        l1 = self.cls(o)
        l2 = self.rpl(o)
        a = torch.sigmoid(self.alpha)
        logits = self.rtm(a * l1 + (1 - a) * l2)
        
        return logits
    
    def predict(self, embeddings, lengths=None):
        """Predict labels for embeddings"""
        logits = self.forward(embeddings, lengths)
        return torch.argmax(logits, dim=-1)

# =========================================================
# LOAD HSLN ROLE CLASSIFIER
# =========================================================
print("\n📚 Loading HSLN role classifier...")

if not os.path.isdir(ROLE_MODEL_PATH):
    raise FileNotFoundError(f"❌ {ROLE_MODEL_PATH} folder not found.")

# Check files
weights_path = os.path.join(ROLE_MODEL_PATH, "pytorch_model.bin")
prototypes_path = os.path.join(ROLE_MODEL_PATH, "prototypes.pt")

if not os.path.exists(weights_path):
    raise FileNotFoundError(f"❌ {weights_path} not found")
if not os.path.exists(prototypes_path):
    raise FileNotFoundError(f"❌ {prototypes_path} not found")

# Load config to get hyperparameters
config_path = os.path.join(ROLE_MODEL_PATH, "config.json")
if os.path.exists(config_path):
    print("   Loading config for hyperparameters...")
    with open(config_path, 'r') as f:
        config_dict = json.load(f)
    
    embedding_dim = config_dict.get('embedding_dim', 768)
    lstm_hidden = config_dict.get('lstm_hidden', 384)
    dropout = config_dict.get('dropout', 0.3)
    rtm_lambda = config_dict.get('rtm_lambda', 0.05)
    
    print(f"   Config: embedding_dim={embedding_dim}, lstm_hidden={lstm_hidden}")
else:
    print("   No config found, using defaults")
    embedding_dim = 768
    lstm_hidden = 384
    dropout = 0.3
    rtm_lambda = 0.05

# Initialize model with CORRECT num_labels
print(f"   Initializing model with {NUM_LABELS} labels...")
role_model = HSLNModel(
    embedding_dim=embedding_dim,
    lstm_hidden=lstm_hidden,
    num_labels=NUM_LABELS,  # HARDCODED 13
    dropout=dropout,
    rtm_lambda=rtm_lambda
)

# Load weights
print("   Loading model weights...")
state_dict = torch.load(weights_path, map_location=device)

# Remove 'prototypes' from state_dict (we load it separately)
if 'prototypes' in state_dict:
    del state_dict['prototypes']

# Load state dict
try:
    missing_keys, unexpected_keys = role_model.load_state_dict(state_dict, strict=False)
    
    if missing_keys:
        # Filter out 'prototypes' from missing keys (expected)
        missing_keys = [k for k in missing_keys if k != 'prototypes']
        if missing_keys:
            print(f"   ⚠️  Missing keys: {missing_keys}")
    
    if unexpected_keys:
        print(f"   ⚠️  Unexpected keys: {unexpected_keys}")
    
    print("   ✓ Weights loaded successfully")
    
except Exception as e:
    print(f"   ❌ Error loading weights: {e}")
    raise

# Load prototypes separately
print("   Loading prototypes...")
prototypes = torch.load(prototypes_path, map_location=device)
role_model.prototypes = prototypes
print(f"   ✓ Prototypes loaded: shape {prototypes.shape}")

# Move to device and set eval mode
role_model.to(device)
role_model.eval()

print("✅ HSLN role classifier loaded successfully!")
print(f"   Architecture:")
print(f"     - Embedding dim: {embedding_dim}")
print(f"     - LSTM hidden: {lstm_hidden}")
print(f"     - Num labels: {NUM_LABELS}")
print(f"     - Labels: {', '.join(LABELS[:5])}... ({NUM_LABELS} total)")

# =========================================================
# LOAD InLegalBERT FOR ROLE CLASSIFICATION
# =========================================================
print("\n📚 Loading InLegalBERT for embeddings...")
base_model_name = "law-ai/InLegalBERT"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModel.from_pretrained(base_model_name).to(device)
base_model.eval()
print("✅ InLegalBERT loaded")

# =========================================================
# ROLE CLASSIFICATION FUNCTION
# =========================================================
@torch.no_grad()
def classify_roles(sentences, batch_size=BATCH_SIZE):
    """Classify rhetorical roles using HSLN"""
    if not sentences:
        return []
    
    all_predictions = []
    
    for i in range(0, len(sentences), batch_size):
        batch_sents = sentences[i:i+batch_size]
        
        # Embed sentences
        inputs = base_tokenizer(
            batch_sents,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)
        
        embeddings = base_model(**inputs).last_hidden_state.mean(dim=1)
        embeddings_batch = embeddings.unsqueeze(1)
        lengths = torch.ones(len(batch_sents), dtype=torch.long).to(device)
        
        # Predict
        predictions = role_model.predict(embeddings_batch, lengths)
        batch_preds = predictions[:, 0].cpu().tolist()
        all_predictions.extend(batch_preds)
    
    return [LABELS[pred] for pred in all_predictions]

# =========================================================
# LOAD LED MODEL
# =========================================================
print("\n📚 Loading fine-tuned LED...")

if not os.path.isdir(LED_MODEL_PATH):
    raise FileNotFoundError(f"❌ {LED_MODEL_PATH} folder not found.")

led_tokenizer = AutoTokenizer.from_pretrained(LED_MODEL_PATH)
led_model = AutoModelForSeq2SeqLM.from_pretrained(LED_MODEL_PATH).to(device)
led_model.eval()

print("✅ LED loaded successfully")

# =========================================================
# LOAD InLegalBERT FOR MCS
# =========================================================
print("\n📚 Loading InLegalBERT for MCS embeddings...")
embed_tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
embed_model = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device)
embed_model.eval()
print("✅ MCS embedder loaded")

# =========================================================
# MCS EMBEDDING FUNCTION
# =========================================================
@torch.no_grad()
def embed_sentences(sentences, batch_size=BATCH_SIZE):
    """Embed sentences for MCS selection"""
    if not sentences:
        return np.zeros((0, 768))
    
    all_embeddings = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        
        inputs = embed_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        outputs = embed_model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        pooled = (outputs * mask).sum(1) / mask.sum(1)
        all_embeddings.append(pooled.cpu().numpy())
    
    return np.vstack(all_embeddings)

# =========================================================
# COMPUTE ROLE IMPORTANCE
# =========================================================
def compute_role_importance(train_json, save_path):
    """Compute importance scores for each role"""
    print("\n🔎 Computing role importance from training data...")
    
    with open(train_json) as f:
        data = json.load(f)
    
    data = data[:MAX_SAMPLES]
    
    role_counts = defaultdict(int)
    role_summary_counts = defaultdict(int)
    
    for item in tqdm(data, desc="Processing training samples"):
        doc_sents = sent_tokenize(item["judgment"])
        sum_sents = sent_tokenize(item["reference_summary"])
        
        if not doc_sents or not sum_sents:
            continue
        
        # Classify roles
        roles = classify_roles(doc_sents)
        
        # Get embeddings
        doc_emb = embed_sentences(doc_sents)
        sum_emb = embed_sentences(sum_sents)
        
        # Match summary to document
        for s_emb in sum_emb:
            sims = cosine_similarity([s_emb], doc_emb)[0]
            best_idx = np.argmax(sims)
            role_summary_counts[roles[best_idx]] += 1
        
        for r in roles:
            role_counts[r] += 1
    
    # Calculate importance
    importance = {}
    for role in role_counts:
        importance[role] = role_summary_counts[role] / max(role_counts[role], 1)
    
    # Normalize
    if importance:
        max_val = max(importance.values())
        importance = {k: v/max_val for k, v in importance.items()}
    
    # Save
    with open(save_path, "w") as f:
        json.dump(importance, f, indent=2)
    
    print("✅ Importance scores computed and saved")
    print_importance_scores(importance)
    
    return importance

def print_importance_scores(importance):
    """Pretty print importance scores"""
    print("\n📊 Role Importance Scores:")
    print("-" * 60)
    for role, score in sorted(importance.items(), key=lambda x: x[1], reverse=True):
        bar = "█" * int(score * 30)
        print(f"  {role:20s}: {score:.4f} {bar}")
    print("-" * 60)

# =========================================================
# MCS SELECTION
# =========================================================
def mcs_selection(embeddings, indices, k):
    """Mean Cosine Similarity selection"""
    if len(indices) <= k:
        return indices
    
    centroid = embeddings.mean(axis=0, keepdims=True)
    sims = cosine_similarity(embeddings, centroid).squeeze()
    top_indices = np.argsort(sims)[-k:]
    
    return [indices[i] for i in top_indices]

# =========================================================
# ROLE-AWARE SELECTION
# =========================================================
def role_aware_selection(doc, importance, total_k=TOP_K):
    """Select sentences using role-aware MCS"""
    sentences = sent_tokenize(doc)
    if not sentences:
        return ""
    
    # Classify roles
    roles = classify_roles(sentences)
    
    # Get embeddings
    embeddings = embed_sentences(sentences)
    
    # Group by role
    role_groups = defaultdict(list)
    for i, role in enumerate(roles):
        role_groups[role].append(i)
    
    selected = []
    
    # Select from each role
    for role, idxs in role_groups.items():
        k_r = max(1, int(total_k * importance.get(role, 0.01)))
        k_r = min(k_r, len(idxs))
        
        role_embs = embeddings[idxs]
        chosen = mcs_selection(role_embs, idxs, k_r)
        selected.extend(chosen)
    
    selected = sorted(selected)
    return " ".join([sentences[i] for i in selected])

# =========================================================
# GENERATE SUMMARY
# =========================================================
@torch.no_grad()
def generate_summary(text):
    """Generate summary using LED"""
    inputs = led_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)
    
    global_attention_mask = torch.zeros_like(inputs["attention_mask"])
    global_attention_mask[:, 0] = 1
    
    summary_ids = led_model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        global_attention_mask=global_attention_mask,
        max_length=512,
        num_beams=4,
        no_repeat_ngram_size=3
    )
    
    return led_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# =========================================================
# EVALUATION
# =========================================================
def evaluate_model(preds, refs):
    """Evaluate using ROUGE and BERTScore"""
    print("\n📊 Computing evaluation metrics...")
    
    rouge = evaluate.load("rouge")
    bertscore = evaluate.load("bertscore")
    
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    
    print("   Computing BERTScore (this may take a while)...")
    bert_scores = bertscore.compute(
        predictions=preds,
        references=refs,
        model_type="roberta-large",
        lang="en"
    )
    
    return {
        "ROUGE-1": rouge_scores["rouge1"],
        "ROUGE-2": rouge_scores["rouge2"],
        "ROUGE-L": rouge_scores["rougeL"],
        "BERTScore-F1": np.mean(bert_scores["f1"])
    }

# =========================================================
# MAIN PIPELINE
# =========================================================
def main():
    print("\n" + "="*70)
    print("🚀 LEGAL SUMMARIZATION PIPELINE WITH HSLN")
    print("="*70)
    
    # Get importance scores
    if os.path.exists(IMPORTANCE_FILE):
        print(f"\n📂 Loading existing importance scores from {IMPORTANCE_FILE}")
        with open(IMPORTANCE_FILE) as f:
            importance = json.load(f)
        print_importance_scores(importance)
    else:
        print(f"\n📂 Computing new importance scores...")
        importance = compute_role_importance(TRAIN_JSON, IMPORTANCE_FILE)
    
    # Load test data
    print(f"\n📂 Loading test data from {TEST_JSON}...")
    with open(TEST_JSON) as f:
        test_data = json.load(f)
    print(f"   Found {len(test_data)} test samples")
    
    # Process
    print("\n🔄 Processing test samples...")
    print(f"   Selecting top {TOP_K} sentences per document using role-aware MCS")
    
    predictions = []
    references = []
    
    for item in tqdm(test_data, desc="Generating summaries"):
        filtered = role_aware_selection(item["judgment"], importance, TOP_K)
        summary = generate_summary(filtered)
        predictions.append(summary)
        references.append(item["reference_summary"])
    
    # Evaluate
    results = evaluate_model(predictions, references)
    
    # Display results
    print("\n" + "="*70)
    print("📊 FINAL RESULTS")
    print("="*70)
    for k, v in results.items():
        print(f"  {k:20s}: {v:.4f}")
    print("="*70)
    
    # Save
    output_data = {
        "metrics": results,
        "config": {
            "top_k": TOP_K,
            "max_training_samples": MAX_SAMPLES,
            "batch_size": BATCH_SIZE,
            "num_test_samples": len(test_data),
            "num_labels": NUM_LABELS
        },
        "importance_scores": importance,
        "predictions": predictions,
        "references": references
    }
    
    with open(OUTPUT_FILE, "w") as f:
        json.dump(output_data, f, indent=2)
    
    print(f"\n💾 Results saved to {OUTPUT_FILE}")
    print("\n✅ PIPELINE COMPLETE!")
    print("="*70)

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Pipeline interrupted by user")
    except Exception as e:
        print(f"\n\n❌ Error: {str(e)}")
        import traceback
        traceback.print_exc()

🖥 Using device: cuda

📚 Loading HSLN role classifier...
   Loading config for hyperparameters...
   Config: embedding_dim=768, lstm_hidden=384
   Initializing model with 13 labels...
   Loading model weights...
   ✓ Weights loaded successfully
   Loading prototypes...
   ✓ Prototypes loaded: shape torch.Size([13, 768])
✅ HSLN role classifier loaded successfully!
   Architecture:
     - Embedding dim: 768
     - LSTM hidden: 384
     - Num labels: 13
     - Labels: PREAMBLE, FAC, RLC, ISSUE, ARG_PETITIONER... (13 total)

📚 Loading InLegalBERT for embeddings...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded

📚 Loading fine-tuned LED...
✅ LED loaded successfully

📚 Loading InLegalBERT for MCS embeddings...
✅ MCS embedder loaded

🚀 LEGAL SUMMARIZATION PIPELINE WITH HSLN

📂 Computing new importance scores...

🔎 Computing role importance from training data...


Processing training samples: 100%|██████████████████████████████████████████████████| 3000/3000 [24:34<00:00,  2.04it/s]


✅ Importance scores computed and saved

📊 Role Importance Scores:
------------------------------------------------------------
  PRE_RELIED          : 1.0000 ██████████████████████████████
  PRE_NOT_RELIED      : 0.7154 █████████████████████
  ARG_PETITIONER      : 0.6367 ███████████████████
  RLC                 : 0.6228 ██████████████████
  ANALYSIS            : 0.4431 █████████████
  RATIO               : 0.3960 ███████████
  FAC                 : 0.3959 ███████████
  ISSUE               : 0.3803 ███████████
  ARG_RESPONDENT      : 0.3780 ███████████
  STA                 : 0.3663 ██████████
  PREAMBLE            : 0.2908 ████████
  RPC                 : 0.2395 ███████
  NONE                : 0.2229 ██████
------------------------------------------------------------

📂 Loading test data from test.json...
   Found 100 test samples

🔄 Processing test samples...
   Selecting top 150 sentences per document using role-aware MCS


Generating summaries: 100%|███████████████████████████████████████████████████████████| 100/100 [37:32<00:00, 22.53s/it]



📊 Computing evaluation metrics...
   Computing BERTScore (this may take a while)...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📊 FINAL RESULTS
  ROUGE-1             : 0.4839
  ROUGE-2             : 0.2519
  ROUGE-L             : 0.2696
  BERTScore-F1        : 0.8533

💾 Results saved to final_results_mcs_hsln.json

✅ PIPELINE COMPLETE!
